# 재료 마스터 정렬과 보관 가이드 보강

Production 재료 마스터를 기존 표준화 기준(`scripts/ingredient_master/load_ingredient_master.py` + `config/ingredient_master_*`)에 맞추고,
그 마스터로 보관 가이드를 다시 적재합니다. 파일을 하나씩 보지 않고 이 노트북 한 곳에서 **무엇을 어떤 순서로 바꾸고 결과가 어떤지** 확인합니다.

**왜 필요했나**

- Production 마스터는 기준 로더가 아니라 `data_pipeline sync-master` 로 만들어졌습니다. 공공 영양성분 DB의 중·소·세분류명(`생것`, 품종명, `감자`(전분의 원료))이
  별칭으로 들어갔고, 별칭 사전은 키가 겹치면 id 가 낮은 재료를 고릅니다. 그래서 `감자 -> 전분`, `고구마 -> 당면`, `Beef short ribs -> 돼지고기` 같은 오매칭이 생겼습니다.
- 육류 부위가 없어 데모 목심 상품의 보관 가이드가 404 였습니다.
- 기준에 있는 재료 7개가 없고, 기준 밖 재료(MFDS 가공 대표식품, 팀 기본 재료)가 기준 문서에 없었습니다.

**바꾸는 것 (순서대로)**

| 단계 | 스크립트 | 설정 |
|---|---|---|
| 1. 육류 부위 추가 | `data_pipeline/scripts/db/add_meat_parts.py` | `config/ingredient_master_meat_parts.csv` |
| 2. 마스터 정렬 (누락 추가, 이름 맞춤) | `data_pipeline/scripts/db/align_ingredient_master.py` | `config/ingredient_master_alignment.csv` |
| 3. 별칭을 검토된 것만 남김 | `data_pipeline/scripts/db/reset_ingredient_aliases.py` | `config/ingredient_master_aliases.csv`, `processed_exceptions.csv`, `alignment.csv` |
| 4. 보관 가이드 재적재 | `data_pipeline/scripts/db/load_storage_guideline.py` | `config/foodkeeper_ingredient_child_rules.csv`, `foodkeeper_processed_sources.csv` |

조회 쿼리 `product_storage_guideline.sql` 은 부위에 지침이 없으면 부모 지침을 씁니다. 기준 설정 편입(재료 51개)은
`config/ingredient_master_processed_exceptions.csv` 에 있습니다. 상품·레시피 매핑은 이 노트북에서 바꾸지 않습니다.

## 실행 방법

프로젝트 venv 에는 Jupyter 가 없어서, **레포 루트에서** `uv run --with` 로 그때만 붙여 엽니다. 의존성은 바뀌지 않습니다.

```bash
# 레포 루트에서. 접속 문자열 값은 셸에만 두고 노트북에 적지 않습니다.
export DATABASE_URL=...          # 대상 DB
export DATABASE_URL_KIPIL=...    # 보관 지침 원천 (KIPIL)
uv run --with jupyter jupyter lab notebooks/ingredient_master_storage_alignment.ipynb

# 창 없이 끝까지 실행만 할 때
uv run --with nbconvert --with ipykernel jupyter nbconvert --to notebook --execute --inplace \
  notebooks/ingredient_master_storage_alignment.ipynb
```

- 대상 DB 접속 문자열은 환경변수로만 넘깁니다. `NB_TARGET_URL_ENV`(기본 `DATABASE_URL`), 보관 지침 원천(KIPIL)은 `NB_SOURCE_URL_ENV`(기본 `DATABASE_URL_KIPIL`).
- **기본은 dry-run** 입니다. 어느 DB에 연결해도 쓰지 않습니다.
- `NB_APPLY=1` 이면 적용합니다. **localhost DB에만** 적용하며, Production 적용은 이 노트북이 아니라 각 스크립트의 `--confirm-production` 절차로 합니다.
- 아래 출력은 **로컬 Production 사본(2026-09-23 읽기 전용 덤프)을 복제한 DB에 `NB_APPLY=1` 로 실행한 결과**입니다.

In [1]:
import csv
import json
import os
import subprocess
import sys
import tempfile
from collections import Counter
from pathlib import Path
from urllib.parse import urlsplit

import psycopg

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
TARGET_ENV = os.environ.get("NB_TARGET_URL_ENV", "DATABASE_URL")
SOURCE_ENV = os.environ.get("NB_SOURCE_URL_ENV", "DATABASE_URL_KIPIL")
APPLY = os.environ.get("NB_APPLY") == "1"
REPORTS = Path(tempfile.mkdtemp(prefix="master_alignment_"))

for name in (TARGET_ENV, SOURCE_ENV):
    if not os.environ.get(name):
        raise RuntimeError(f"{name} 가 비어 있습니다. 접속 문자열 값은 출력하지 않습니다.")
if APPLY and urlsplit(os.environ[TARGET_ENV]).hostname not in {"localhost", "127.0.0.1"}:
    raise RuntimeError("NB_APPLY=1 은 localhost DB에서만 씁니다. Production 은 스크립트의 확인 문자열 절차를 쓰세요.")


def run(module: str, *args: str) -> None:
    """스크립트를 대상 DB에 실행하고 출력을 보여 줍니다. APPLY 일 때만 --apply 를 붙입니다."""
    command = [
        sys.executable,
        "-m",
        f"scripts.db.{module}",
        "--env-file",
        os.devnull,
        "--target-url-env",
        TARGET_ENV,
        "--target",
        "local",
        *args,
    ]
    if APPLY:
        command.append("--apply")
    result = subprocess.run(command, cwd=PROJECT_ROOT / "data_pipeline", text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(result.stderr[-2000:])
    print(result.stdout.replace(str(REPORTS), "<reports>"))


def query(sql: str, params: tuple = ()) -> list[tuple]:
    with psycopg.connect(os.environ[TARGET_ENV]) as connection, connection.cursor() as cursor:
        cursor.execute(sql, params)
        return cursor.fetchall()


print(f"APPLY={APPLY}")

APPLY=True


## 1. 육류 부위 추가

기준 문서의 고기 부위 규칙 그대로입니다. 대표식품(돼지고기, 소고기, 닭고기, 양고기)을 부모로 두고 공통 부위 29개를 SMALL child 로 둡니다.
이름은 K-FIND 원천 표기(`목심(목심살)` 의 `목심`)를 씁니다. 이미 있는 돼지고기 `목심`(1029)은 그대로 둡니다.

In [2]:
run("add_meat_parts")

mode=APPLIED target=local
configured=29 already_present=1 to_insert=28
  + 돼지고기 > 갈비
  + 돼지고기 > 뒷다리
  + 돼지고기 > 등심
  + 돼지고기 > 사태
  + 돼지고기 > 삼겹살
  + 돼지고기 > 안심
  + 돼지고기 > 앞다리
  + 소고기 > 갈비
  + 소고기 > 등심
  + 소고기 > 목심
  + 소고기 > 사태
  + 소고기 > 설도
  + 소고기 > 안심
  + 소고기 > 앞다리
  + 소고기 > 양지
  + 소고기 > 우둔
  + 소고기 > 채끝
  + 닭고기 > 가슴
  + 닭고기 > 날개
  + 닭고기 > 넓적다리
  + 닭고기 > 다리
  + 닭고기 > 목
  + 닭고기 > 아랫다리
  + 닭고기 > 껍질
  + 어린양고기 > 갈비
  + 어린양고기 > 다리
  + 어린양고기 > 어깨
  + 양고기 > 다리
inserted=28



## 2. 마스터 정렬

기준 로더 결과와 Production 마스터를 식별키로 비교해 나온 차이 중, 기준에만 있는 재료 6개를 추가하고 이름 3개(부침가루, 조미김, 떡갈비)를 기준에 맞춥니다.
가공 재료 식별키는 Production 형식(`K-FIND-P:<대분류>:<대표코드>:<대표명>`, `TEAM-BASIC:<이름>`)을 씁니다. 떡갈비는 MFDS 대표식품 분쇄가공육 행의 이름을 바꿉니다.
Production 에만 있고 참조되는 재료 중 레시피 재료로 쓰이는 51개(팀 기본 재료 9, 가공 재료 42)는 DB가 아니라 기준 설정에 편입했습니다.
원재료와 같은 재료(밀가루, 간장 등 13), 분류명(식물성유지, 향신료 등 10), 완제품(피자, 떡볶이 등 21)은 편입하지 않고,
Production 행은 참조가 있어 그대로 둔 채 매핑 재생성 때 정리합니다.

In [3]:
run("align_ingredient_master")

mode=APPLIED target=local
configured=9 to_apply=9
  INSERT 꽈리고추 (K-FIND:06:06018:고추:MIDDLE:꽈리고추)
  INSERT 마늘가루 (K-FIND:18:18053:마늘:MIDDLE:마늘가루)
  INSERT 양파가루 (K-FIND:18:18056:양파:MIDDLE:양파가루)
  INSERT 굴소스 (TEAM-BASIC:굴소스)
  INSERT 데리야끼소스 (TEAM-BASIC:데리야끼소스)
  INSERT 맛술 (TEAM-BASIC:맛술)
  RENAME 부침가루 (K-FIND-P:16:16708:부침가루/튀김가루/믹스)
  RENAME 조미김 (K-FIND-P:20:20401:김)
  RENAME 떡갈비 (K-FIND-P:17:17502:분쇄가공육)



## 3. 별칭 정리

별칭은 **진짜 동의어**(계란 = 달걀, 케첩 = 토마토케첩)만 둡니다. 중분류는 기준대로 대표식품에 합치거나 승격된 자식으로 두고, 소·세분류명은 재료 이름이 아닙니다.
검토된 목록 밖의 별칭은 지우고 리포트로 남깁니다. 이미 만들어진 상품·레시피 매핑은 바뀌지 않습니다.

In [4]:
report = REPORTS / "aliases_removed.csv"
run("reset_ingredient_aliases", "--report", str(report))
removed = list(csv.DictReader(report.open(encoding="utf-8")))
print("지운 별칭이 많은 재료:", Counter(row["name"] for row in removed).most_common(8))
bad = {("전분", "감자"), ("당면", "고구마"), ("찹쌀", "백미"), ("꿀", "토종")}
print(
    "오매칭을 만들던 별칭 예:",
    [f"{r['name']}:{r['removed_alias']}" for r in removed if (r["name"], r["removed_alias"]) in bad],
)

mode=APPLIED target=local
ingredients_changed=918 aliases_removed=3505
report=<reports>/aliases_removed.csv

지운 별칭이 많은 재료: [('소고기', 59), ('멥쌀', 50), ('새우류', 36), ('옥수수', 34), ('돔류', 34), ('돼지고기', 33), ('가자미류', 30), ('고둥류', 30)]
오매칭을 만들던 별칭 예: ['찹쌀:백미', '당면:고구마', '전분:감자', '꿀:토종']


## 4. 보관 가이드 재적재

KIPIL 보관 지침 935행을 다음 순서로 선별합니다.

1. 원재료에 붙은 가공·조리 원천(FoodKeeper 카테고리 11-14, 16, 17)을 뺍니다. 대상에 이미 있는 같은 행도 지웁니다.
2. 부위 규칙(원천 항목 단위로 검토)을 따라 부위 child 로 옮깁니다. 원래 매핑보다 우선합니다.
3. 그래도 한 (재료, 장소, 상황)에 기간이 다른 원천이 남으면 가장 짧은 기간을 고르고 결정 리포트로 남깁니다.

적재는 upsert 입니다. 먼저 원재료에 붙어 있던 가공 원천 행을 지우고(아래 출력 `pruned_processed=1`), 나머지 기존 행은 덮어씁니다(`updated=472`). 대표 원천이 바뀐 재료는 기존 행 내용도 바뀌므로 Production 적용 전에 이 출력을 확인하세요.

In [5]:
decisions_path = REPORTS / "storage_decisions.jsonl"
run("load_storage_guideline", "--source-url-env", SOURCE_ENV, "--report", str(decisions_path))
decisions = [json.loads(line) for line in decisions_path.open(encoding="utf-8")]
ids = [d["ingredient_id"] for d in decisions]
names = dict(query("SELECT ingredient_id, name FROM ingredient WHERE ingredient_id = ANY(%s)", (ids,)))
print(f"가장 짧은 기간으로 고른 조합 {len(decisions)}개. 육류 예:")
for d in decisions:
    if names.get(d["ingredient_id"]) in {"돼지고기", "소고기", "닭고기"}:
        slot = f"{d['storage_location']}·{d['storage_context']}"
        print(
            f"  {names[d['ingredient_id']]} {slot} -> {d['chosen']} {d['chosen_duration']}",
            f"(나머지 {len(d['others'])}개)",
        )

mode=APPLIED target=local
source_rows=935 processed_dropped=10 moved_to_child=34
accepted_rows=609 accepted_ingredients=236
shortest_picked_groups=122
report=<reports>/storage_decisions.jsonl
inserted=137 updated=472 pruned_processed=1

가장 짧은 기간으로 고른 조합 122개. 육류 예:
  닭고기 냉동·구매후 -> Ground turkey or chicken 3-4 개월 (나머지 3개)
  돼지고기 냉동·구매후 -> Pork / ground 3-4 개월 (나머지 4개)
  돼지고기 냉장·구매후 -> Pork / ground 1-2 일 (나머지 4개)
  소고기 냉동·구매후 -> Beef / ground 3-4 개월 (나머지 4개)
  소고기 냉장·구매후 -> Beef / ground 1-2 일 (나머지 4개)


## 5. 결과 확인 (읽기 전용)

In [6]:
print("부위 계층")
for parent, children in query("""
    SELECT p.name, string_agg(c.name, ', ' ORDER BY c.name)
    FROM ingredient c JOIN ingredient p ON p.ingredient_id = c.parent_ingredient_id
    WHERE c.source_identity_key LIKE 'K-FIND:09:%%:SMALL:%%' GROUP BY p.name ORDER BY p.name"""):
    print(f"  {parent} > {children}")

print("\n마스터 정렬로 추가·변경한 재료")
for row in query("""
    SELECT i.name, p.name, i.aliases, i.is_raw_material FROM ingredient i
    LEFT JOIN ingredient p ON p.ingredient_id = i.parent_ingredient_id
    WHERE i.name IN ('꽈리고추', '마늘가루', '양파가루', '굴소스', '데리야끼소스',
                     '맛술', '떡갈비', '부침가루', '조미김')
    ORDER BY 1"""):
    print("  ", row)

print("\n남은 별칭")
for name, aliases in query("SELECT name, aliases FROM ingredient WHERE cardinality(aliases) > 0 ORDER BY name"):
    print(f"  {name}: {', '.join(aliases)}")

부위 계층
  닭고기 > 목, 가슴, 껍질, 날개, 다리, 넓적다리, 아랫다리
  소고기 > 갈비, 등심, 목심, 사태, 설도, 안심, 양지, 우둔, 채끝, 앞다리
  양고기 > 다리
  돼지고기 > 갈비, 등심, 목심, 사태, 안심, 뒷다리, 삼겹살, 앞다리
  어린양고기 > 갈비, 다리, 어깨

마스터 정렬로 추가·변경한 재료
   ('맛술', None, ['미림', '미향', '요리술'], False)
   ('굴소스', None, [], False)
   ('떡갈비', None, [], False)
   ('조미김', None, ['구운김'], False)
   ('꽈리고추', '고추', [], True)
   ('마늘가루', None, [], True)
   ('부침가루', None, ['튀김가루'], False)
   ('양파가루', None, [], True)
   ('데리야끼소스', None, ['데리야끼', '데리야키'], False)

남은 별칭
  달걀: 계란
  맛술: 미림, 미향, 요리술
  버터: 가염버터, 무염버터
  식초: 사과식초, 양조식초, 현미식초
  치즈: 모짜렐라치즈, 모차렐라, 슬라이스치즈, 체다치즈, 피자치즈
  카레: 카레가루, 커리
  후추: 후추가루, 후춧가루
  소고기: 쇠고기
  조미김: 구운김
  마요네즈: 마요
  부침가루: 튀김가루
  브로콜리: 브로컬리
  올리브유: 올리브오일
  양송이버섯: 양송이
  유채씨기름: 카놀라유
  토마토케첩: 케찹, 케챱, 케첩
  데리야끼소스: 데리야끼, 데리야키


In [7]:
total, nuggets, beef_on_pork = query("""
    SELECT count(*),
           count(*) FILTER (WHERE source_food_name = 'Chicken nuggets, patties' AND ingredient_id = 401),
           count(*) FILTER (WHERE source_food_name = 'Beef' AND ingredient_id = 404)
    FROM storage_guideline""")[0]
print(f"보관 지침 {total}행, 생닭에 붙은 너겟 지침 {nuggets}행, 돼지고기에 붙은 소고기 지침 {beef_on_pork}행")

# 데모 목심 상품(컬리 10153238)의 보관 가이드. 서빙과 같은 SQL 을 씁니다.
catalog = PROJECT_ROOT / "recsys_sql/src/recsys_sql/queries/openLeeWorld"
sql = (catalog / "product_storage_guideline.sql").read_text(encoding="utf-8")
sql = sql.replace(":product_id", "%(product_id)s")
product_id = query("SELECT product_id FROM product WHERE source_product_id = '10153238'")[0][0]
with psycopg.connect(os.environ[TARGET_ENV]) as connection, connection.cursor() as cursor:
    cursor.execute(sql, {"product_id": product_id})
    for row in cursor.fetchall():
        print(f"  {row[1]} | 재료 {row[4]} | {row[5]}·{row[6]} {row[10]}")

보관 지침 609행, 생닭에 붙은 너겟 지침 0행, 돼지고기에 붙은 소고기 지침 0행
  [하이포크] 무항생제 한돈 목살 구이용 300g (냉장) | 재료 목심 | 냉동·구매후 3-4 개월
  [하이포크] 무항생제 한돈 목살 구이용 300g (냉장) | 재료 목심 | 냉장·구매후 1-2 일


## Production 적용 순서

#26, #27 머지 뒤에 아래 순서로 실행합니다. 각 스크립트는 먼저 dry-run 으로 확인하고, `--apply --confirm-production <토큰>` 으로 적용합니다.

1. `add_meat_parts` (`ADD_MEAT_PARTS_V1`)
2. `align_ingredient_master` (`ALIGN_INGREDIENT_MASTER_V1`)
3. `reset_ingredient_aliases` (`RESET_INGREDIENT_ALIASES_V1`)
4. `load_storage_guideline` (`LOAD_STORAGE_GUIDELINE_V1`)
5. 데모 시드 검증 `uv run data-pipeline seed-scenario` (dry-run, 8항목 통과 확인)

남은 일: 정렬된 마스터로 상품·레시피 매핑을 다시 만드는 작업은 데모 뒤에 따로 합니다.